# Hyper-optimisation — Adult Income (UCI)

Ce notebook optimise les hyperparamètres des meilleurs modèles avec **Optuna** via le module `HyperparameterTuner`, puis ré-évalue les pipelines optimisés avec `ModelEvaluator`.

Pipeline : `DataLoader` → `PreprocessingPipeline` → `ModelRegistry` → `HyperparameterTuner` (Optuna) → `ModelEvaluator`.

In [ ]:
# Recharge automatique des modules src/ modifiés sans restart du kernel
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Permet d'importer le package src/ depuis le dossier notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from adult_income import (
    DataLoader,
    HyperparameterTuner,
    ModelEvaluator,
    ModelRegistry,
    PreprocessingPipeline,
)

print("Imports OK")

## 1 — Données & preprocessing

In [ ]:
# --- Données : split train/test stratifié ---
loader = DataLoader()
X_train, X_test, y_train, y_test = loader.load_and_split()

pp = PreprocessingPipeline()

# Encoder la cible (<=50K -> 0, >50K -> 1)
y_train_enc = pp.fit_transform_target(y_train)
y_test_enc = pp.transform_target(y_test)

print(f"Train : {X_train.shape[0]:,} obs | Test : {X_test.shape[0]:,} obs")

## 2 — Optimisation Optuna (`HyperparameterTuner`)

On optimise **tous les modèles possibles** (ceux qui exposent un espace de recherche). Pour chaque modèle, `tune()` lance une étude Optuna qui maximise la **PR-AUC** (`average_precision`) en cross-validation, puis on reconstruit le pipeline avec les meilleurs paramètres. La PR-AUC est privilégiée au `roc_auc` car elle reflète mieux la performance sur la classe minoritaire (revenus > 50K).

**Persistance** — en passant `storage=<dossier>`, chaque étude est sauvegardée dans `artifacts/optuna_ap` :
- `optuna_studies.db` : toutes les trials (SQLite) → permet de **reprendre** une optimisation interrompue et de recharger les graphiques plus tard ;
- `best_params.json` : récapitulatif léger des meilleurs hyperparamètres, réutilisable ailleurs (app Streamlit) sans rejouer Optuna.

In [ ]:
# On optimise TOUS les modèles possibles, c.-à-d. ceux qui ont un espace de
# recherche défini dans le tuner. dummy / stacking n'en ont pas et seraient
# rejetés par tune(), on les exclut donc automatiquement.
MODELS_TO_TUNE = HyperparameterTuner.tunable_models()
print(f"Modèles à optimiser ({len(MODELS_TO_TUNE)}) : {MODELS_TO_TUNE}")

# scoring="average_precision" => optimisation sur la PR-AUC (pr_auc), adaptée au
# déséquilibre des classes du dataset Adult.
# storage=<dossier> => sauvegarde des études (SQLite) + best_params.json.
# Si on relance la cellule, les études déjà complètes ne sont PAS recalculées.
tuner = HyperparameterTuner(
    n_trials=50, cv=5, scoring="average_precision", n_jobs=4, storage=ROOT / "artifacts" / "optuna_ap"
)

tuned_pipelines = {}
for name in MODELS_TO_TUNE:
    print(f"\nTuning {name}...")
    estimator = ModelRegistry.get(name)
    pipeline = pp.build_pipeline(estimator)
    best_params = tuner.tune(name, pipeline, X_train, y_train_enc)
    print(f"  Best params: {best_params}")

    # Reconstruire le pipeline avec les meilleurs paramètres
    tuned_est = ModelRegistry.get(name)
    tuned_est.set_params(**best_params)
    tuned_pipelines[f"{name}_tuned"] = pp.build_pipeline(tuned_est)

In [ ]:
# Récapitulatif des meilleurs scores CV par modèle
tuner.summary()

## 3 — Visualisations Optuna

Graphes interactifs (plotly) issus des études : historique d'optimisation, importance des paramètres, coordonnées parallèles.

In [ ]:
tuner.plot_optimization_history("xgboost")

In [ ]:
tuner.plot_param_importances("xgboost")

In [ ]:
tuner.plot_parallel_coordinate("random_forest")

## 4 — Évaluation finale des modèles optimisés

In [ ]:
import matplotlib.pyplot as plt

evaluator = ModelEvaluator(cv=5, random_state=42)
for name, pipeline in tuned_pipelines.items():
    evaluator.evaluate_on_test(name, pipeline, X_train, y_train_enc, X_test, y_test_enc)

evaluator.test_comparison_table()

In [ ]:
evaluator.plot_metric_heatmap()
plt.show()

In [ ]:
evaluator.plot_roc_curves(X_test, y_test_enc)
plt.show()

## 5 — Focus sur le meilleur modèle optimisé

In [ ]:
best = evaluator.test_comparison_table().iloc[0]["model"]
print(f"Meilleur modèle optimisé (roc_auc test) : {best}")

evaluator.print_classification_report(best, tuned_pipelines[best], X_test, y_test_enc)

In [ ]:
evaluator.plot_confusion_matrix(best, X_test, y_test_enc)
plt.show()

## 6 — Recharger une optimisation sauvegardée

Tout est sur disque dans `artifacts/optuna`. Dans une nouvelle session (kernel redémarré, autre notebook, app Streamlit), on récupère les résultats **sans relancer** les trials.

In [ ]:
# Récap léger : lit juste best_params.json (pas besoin d'Optuna)
best = HyperparameterTuner.load_best_params(ROOT / "artifacts" / "optuna_ap")
for name, info in best.items():
    print(f"{name:20s} {info['scoring']}={info['best_value']:.4f}  {info['best_params']}")

In [ ]:
# Recharger une étude complète depuis SQLite pour ré-afficher les graphiques
tuner2 = HyperparameterTuner(
    storage=ROOT / "artifacts" / "optuna_ap",
    scoring="average_precision",      # doit matcher le scoring utilisé au tuning
)
print("Études sauvegardées :", tuner2.list_saved_studies())
tuner2.load_study("xgboost")          # nom de base, pas "xgboost__average_precision"
tuner2.plot_optimization_history("xgboost")